# Phase 1 — DPO

Entraîne l'adapter DPO sur les paires PKU-SafeRLHF + UltraFeedback.

**Setup Kaggle** : GPU T4 x1, Internet On.  
**Durée approximative** : ~10 h sur T4 (1 époque, 1 029 steps).

À la fin : zipper `results/dpo_model/` et **créer un Kaggle Dataset** depuis l'output (`dpo_model.zip`) — il sera attaché aux notebooks suivants.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# Optionnel : pour la génération synthétique (data/generate_synthetic.py)
# from kaggle_secrets import UserSecretsClient
# os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")

In [ ]:
!pip install -q -U bitsandbytes transformers==4.46.3 trl==0.12.0 peft==0.14.0 \
    accelerate==1.2.0 datasets==3.2.0 sentence-transformers faiss-cpu \
    anthropic openai pyarrow==17.0.0 tqdm

In [ ]:
!rm -rf /kaggle/working/adl
!git clone https://github.com/FeelTheFloww/adl-ethics.git /kaggle/working/adl
%cd /kaggle/working/adl

In [ ]:
!python data/prepare_preferences.py \
    --n_pku 15000 --n_ultra 5000 \
    --out_path data/preferences.jsonl

In [ ]:
!python training/train_dpo.py \
    --data_path data/preferences.jsonl \
    --output_dir results/dpo_model \
    --batch_size 1 --grad_accum 16 --max_length 384

## Export

Une fois l'entraînement terminé, zippe l'adapter pour l'exporter en Kaggle Dataset.

In [ ]:
import shutil, os
src = "/kaggle/working/adl/results/dpo_model"
out = "/kaggle/working/dpo_model.zip"
shutil.make_archive(out.replace(".zip", ""), "zip", src)
print(f"Zipped -> {out}  ({os.path.getsize(out)//1024//1024} MB)")
print("\nÉtapes suivantes :")
print("  1. Save Version → Save & Run All")
print("  2. Une fois fini, Output → New Dataset depuis dpo_model.zip")
print("  3. Nomme-le par ex. 'adl-dpo-adapter' — il sera attaché aux notebooks 02 et 04")